In [ ]:
#Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv

load_dotenv()  

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [17]:
# define function to call anthro
def chat_with_anthro(model="claude-sonnet-5-5", max_tokens=1024, messages=[], system=None):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system    
    response = client.messages.create(**params
    )
    return [block.text for block in response.content if getattr(block, 'type', None) == 'text'][0]

In [18]:
# define function to add messages
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})
    return messages

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})
    return messages


In [ ]:
messages = []

while True:
    user_inp = input("> ")
    print("> ", user_inp)
    add_user_message(messages, user_inp)
    if not user_inp:
        break
    answer = chat_with_anthro(messages=messages, system="Be precise and concise. Give short answers as much as possible")
    add_assistant_message(messages, answer)
    print("-"*30)
    print(answer)
    print("-"*30)

In [ ]:
messages = []
add_user_message(messages, "Write a poem about the sun and the moon.")

# efficient streaming of the response
with client.messages.stream(model=model, messages=messages, max_tokens=1024) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

In [ ]:
# get the final message from the stream
stream.get_final_message().content[0].text

In [ ]:
# get the out put in certain format

def formatted_anthro(messages, system=None, output_config=None):
    params = {"model": model,
              "max_tokens": 1024,
              "messages": messages}
    if system:
        params["system"] = system
    if output_config:
        params["output_config"] = output_config
    response = client.messages.create(**params)

    return [block.text for block in response.content if getattr(block, 'type', None) == 'text'][0]

response = formatted_anthro(messages=messages, output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "email": {"type": "string"},
                    "plan_interest": {"type": "string"},
                    "demo_requested": {"type": "boolean"},
                },
                "additionalProperties": False,
            },
        }
    })

'{}'

In [43]:
messages = []
add_user_message(messages, "add 10 example of user data in json format")

# get response in certain format
response = formatted_anthro(messages=messages, output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "users": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "email": {"type": "string"},
                                "plan_interest": {"type": "string"},
                                "demo_requested": {"type": "boolean"},
                            },
                            "additionalProperties": False,
                        },
                    },
                },
                "additionalProperties": False,
            },
        }
    })

In [ ]:
import json
json_data = json.loads(response)
print(json_data)